# 混淆矩阵分析：TCM + Bi-normalization

**目标**：用Bi-normalized TCM找出最容易混淆的宣传技术对，为Prompt-A改进提供依据

- **TCM**：最优传输多标签混淆矩阵（Erbani et al. 2024）。对角=min(u_i,v_i)，非对角=deficit×excess/total
- **Bi-normalization**：IPF双向归一化，去除频率偏差，保留纯类别相似性

## ⚠️ 文件格式说明（所有文件统一4列）

| 文件 | 格式 | article_id格式 |
|------|------|:--------------:|
| Gold spans | `article_id \t technique \t start \t end` | 纯数字 |
| Prompt-A 预测 | `article_id \t technique \t start \t end` | 纯数字 |
| Iter-Ens 预测 | `article_id \t technique \t start \t end` | 纯数字 |
| Sup-FT 预测 | `article_id \t technique \t start \t end` | **带`article`前缀** |

统一处理：去掉`article`前缀，全部转为纯数字字符串。

In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================
import os

# Root directory of this repository
BASE = "your/path/here"  # e.g. "/home/user/propaganda-span-detection"

# SemEval-2023 Task 3 development data
SEMEVAL_DEV_DIR = "your/path/here"  # contains en/, po/, ru/ subdirectories

# CheckThat! 2024 Task 3 data
CHECKTHAT_DIR = "your/path/here"

# S3 credentials (or set as environment variables)
os.environ.setdefault("AWS_ACCESS_KEY_ID",     "your-access-key-id")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")
S3_ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET   = "your-s3-bucket"


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams['font.size'] = 9
print('依赖加载完成')

## 1. 配置

In [ ]:
BASE = Path(BASE  # Set BASE in the configuration cell)

# ===== 修改这两个变量切换分析目标 =====
LANG   = 'en'       # 'en', 'pl', 'ru'
METHOD = 'iter_ens' # 'sup_ft', 'prompt_a', 'iter_ens'
# =====================================

TECH_FILE = BASE / 'technique_list/techniques_subtask3.txt'

# Gold spans: article_id(纯数字) \t technique \t start \t end
GOLD_SPANS = {
    'en': BASE / 'data_analy/overlap_analysis_results/en_overlap_articles/labels-subtask-3-spans.txt',
    'pl': BASE / 'data_analy/overlap_analysis_results/po_overlap_articles/labels-subtask-3-spans.txt',
    'ru': BASE / 'data_analy/overlap_analysis_results/ru_overlap_articles/labels-subtask-3-spans.txt',
}

# Run inference文件：全部4列格式 (article_id \t technique \t start \t end)
# 差异只在 article_id 前缀：Sup-FT带'article'，其余是纯数字
PRED_FILES = {
    # Sup-FT: article_id 带 'article' 前缀
    ('en', 'sup_ft'): BASE / 'results/model_tests/semeval_task3_en_dev_predictions_official_format.txt',
    ('pl', 'sup_ft'): BASE / 'results/model_tests/semeval_task3_po_dev_predictions_official_format.txt',
    ('ru', 'sup_ft'): BASE / 'results/model_tests/semeval_task3_ru_dev_predictions_official_format.txt',
    # Prompt-A: article_id 纯数字
    ('en', 'prompt_a'): BASE / 'results/LLM_tests/results_en_results_semeval_task3_dev_en_context_all.tsv',
    ('pl', 'prompt_a'): BASE / 'results/LLM_tests/semeval2023_task3_dev_results_po_po_context_all.tsv',
    ('ru', 'prompt_a'): BASE / 'results/LLM_tests/semeval2023_task3_dev_results_ru_ru_context_all.tsv',
    # Iter-Ens: article_id 纯数字
    ('en', 'iter_ens'): BASE / 'results/LLM_tests/results_en_results_semeval_task3_dev_en_voting_aggressive_all.tsv',
    ('pl', 'iter_ens'): BASE / 'results/LLM_tests/semeval2023_task3_dev_results_po_po_voting_aggressive_all.tsv',
    ('ru', 'iter_ens'): BASE / 'results/LLM_tests/semeval2023_task3_dev_results_ru_ru_voting_aggressive_all.tsv',
}

print(f'目标: LANG={LANG}, METHOD={METHOD}')
gp = GOLD_SPANS[LANG]
pp = PRED_FILES[(LANG, METHOD)]
print(f'Gold 存在={gp.exists()} | {gp.name}')
print(f'Pred 存在={pp.exists()} | {pp.name}')

# 提前建好输出目录
(BASE / 'results/TCM').mkdir(parents=True, exist_ok=True)
print('输出目录已创建: results/TCM/')

## 2. 数据加载

所有文件格式相同：`article_id \t technique \t start \t end`（4列）  
用一个统一函数加载，只需指定是否剥离`article`前缀。

In [ ]:
# 加载技术列表
with open(TECH_FILE) as f:
    TECHNIQUES = [l.strip() for l in f if l.strip()]
TECH2IDX = {t: i for i, t in enumerate(TECHNIQUES)}
TECH_SET  = set(TECHNIQUES)
C = len(TECHNIQUES)
print(f'{C} 个技术，前5个: {TECHNIQUES[:5]}')


def normalize_id(aid):
    """统一去除 'article' 前缀，返回纯数字字符串"""
    return str(aid).replace('article', '').strip()


def load_gold_with_zero(lang, base):
    """
    以 articles/ 目录为权威，返回 {article_id: set of techniques}
    gold=0 的文章返回空 set，确保 FP 被正确计入
    """
    lang_dir = {'en': 'en', 'pl': 'po', 'ru': 'ru'}[lang]
    overlap  = base / f'data_analy/overlap_analysis_results/{lang_dir}_overlap_articles'
    art_dir  = overlap / 'articles'
    spans_f  = overlap / 'labels-subtask-3-spans.txt'

    # Step 1：articles/ 目录 → 权威文章集合
    all_ids = set()
    for f in art_dir.iterdir():
        nid = f.stem.replace('article', '').strip()
        if nid.isdigit():
            all_ids.add(nid)

    # Step 2：初始化全部为 gold=0
    gold = {aid: set() for aid in all_ids}

    # Step 3：从 spans 文件填入真实标注（过滤伪行）
    with open(spans_f, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 4:
                continue
            aid  = normalize_id(parts[0])
            tech = parts[1].strip()
            s, e = parts[2].strip(), parts[3].strip()
            # 过滤伪行：technique为纯数字或start/end为空
            if tech in TECH_SET and not tech.isdigit() and s != '' and e != '':
                if aid in gold:
                    gold[aid].add(tech)

    gold0 = sum(1 for v in gold.values() if not v)
    print(f'  → {len(gold)} 篇文章（含 gold=0: {gold0} 篇）')
    return gold


def build_multihot(gold_d, pred_d, techniques, t2i):
    """
    以 gold 中的文章为行，构建多热矩阵。
    gold=0 的文章也包含（该行全为0），确保评估完整。
    """
    ids = sorted(gold_d.keys())
    n, C = len(ids), len(techniques)
    Yg = np.zeros((n, C))
    Yp = np.zeros((n, C))
    for r, aid in enumerate(ids):
        for t in gold_d.get(aid, set()):
            if t in t2i: Yg[r, t2i[t]] = 1.
        for t in pred_d.get(aid, set()):
            if t in t2i: Yp[r, t2i[t]] = 1.
    return Yg, Yp, ids

def load_spans_file(filepath, tech_set):
    """统一加载4列spans文件：article_id \\t technique \\t start \\t end"""
    d = defaultdict(set)
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 2: continue
            aid  = normalize_id(parts[0])
            tech = parts[1].strip()
            if tech in tech_set:
                d[aid].add(tech)
    print(f'  → {len(d)} 篇文章，{sum(len(v) for v in d.values())} 条记录')
    return dict(d)
print('Functions defined')

In [ ]:
print(f'加载 Gold ({LANG})...')
gold_d = load_gold_with_zero(LANG, BASE)

print(f'加载 Pred ({METHOD})...')
pred_d = load_spans_file(PRED_FILES[(LANG, METHOD)], TECH_SET)

# 检查 article_id 格式是否对齐（至少有一定重叠）
overlap = set(gold_d) & set(pred_d)
print(f'\nGold文章数: {len(gold_d)}, Pred文章数: {len(pred_d)}, 重叠: {len(overlap)}')
if len(overlap) == 0:
    print('⚠️  重叠为0！请检查 article_id 格式是否统一。')
    print('  Gold 示例ID:', list(gold_d)[:3])
    print('  Pred 示例ID:', list(pred_d)[:3])
else:
    print('✅ article_id 格式对齐正常')

# 构建多热矩阵
Yg, Yp, article_ids = build_multihot(gold_d, pred_d, TECHNIQUES, TECH2IDX)

print(f'\n矩阵形状: {Yg.shape}  (文章数 × 技术数)')
print(f'Gold 平均标签数/文章: {Yg.sum(1).mean():.2f}')
print(f'Pred 平均标签数/文章: {Yp.sum(1).mean():.2f}')
print(f'Gold=0 的文章数 (无宣传): {(Yg.sum(1)==0).sum()}')

In [ ]:
# 各技术的 gold / pred 样本数
df_counts = pd.DataFrame({
    'technique': TECHNIQUES,
    'gold_articles': Yg.sum(0).astype(int),
    'pred_articles': Yp.sum(0).astype(int),
}).sort_values('gold_articles', ascending=False)
print('各技术文章数（按Gold降序）:')
print(df_counts.to_string(index=False))

## 3. TCM 计算

In [ ]:
def tcm_contribution(y, yh):
    """
    单实例TCM贡献矩阵（Erbani et al. 2024 Proposition 3）
    - 对角: π*_ii = min(u_i, v_i)  → 正确量
    - 非对角: π*_ij = deficit_i × excess_j / total_deficit  → 混淆量
    """
    ny, nh = y.sum(), yh.sum()
    if ny == 0 or nh == 0:
        return np.zeros((len(y), len(y)))
    u, v = y / ny, yh / nh
    m       = np.minimum(u, v)       # 共同量
    deficit = u - m                  # gold有但pred没有
    excess  = v - m                  # pred有但gold没有
    total   = deficit.sum()          # = excess.sum() by Lemma 1
    pi_diag = np.diag(m)
    if total < 1e-10:
        return pi_diag               # 完美预测
    pi_off  = np.outer(deficit, excess) / total
    np.fill_diagonal(pi_off, 0.0)
    return pi_diag + pi_off


def compute_tcm(Yg, Yp, weighting='one'):
    """TCM = Σ λ_n × π*(y_n, ŷ_n)  （weighting: 'one'|'label'|'pred'）"""
    T = np.zeros((Yg.shape[1],) * 2)
    for i in range(len(Yg)):
        y, yh = Yg[i], Yp[i]
        lam   = 1.0 if weighting == 'one' else (y.sum() if weighting == 'label' else yh.sum())
        if lam == 0: continue
        T += lam * tcm_contribution(y, yh)
    return T


# --- 单元测试 ---
# 完美预测 → 只有对角
_y = np.array([1.,1.,0.])
_pi = tcm_contribution(_y, _y)
assert np.allclose(_pi, np.diag([.5,.5,0.])), '测试失败'
# 完全错误 → 非对角有值
_pi2 = tcm_contribution(np.array([1.,0.,0.]), np.array([0.,1.,0.]))
assert _pi2[0,0] == 0 and _pi2[0,1] > 0, '测试失败'
print('✅ TCM单元测试通过')

print('计算TCM...')
TCM = compute_tcm(Yg, Yp, 'one')
print(f'对角占比（正确量）: {np.trace(TCM)/TCM.sum():.1%}')

## 4. Bi-normalization (IPF)

In [ ]:
def bi_norm(M, eps=1e-8, maxiter=2000, tol=1e-7):
    """IPF/Sinkhorn-Knopp：交替行/列归一化直至收敛（每行每列都=1）"""
    Q = M.copy().astype(float) + eps
    for it in range(maxiter):
        Q /= Q.sum(1, keepdims=True)
        Q /= Q.sum(0, keepdims=True)
        err = max(abs(Q.sum(1) - 1).max(), abs(Q.sum(0) - 1).max())
        if err < tol:
            return Q, it + 1
    return Q, maxiter


def row_norm(M):
    s = M.sum(1, keepdims=True)
    return M / np.where(s == 0, 1, s)


def col_norm(M):
    s = M.sum(0, keepdims=True)
    return M / np.where(s == 0, 1, s)


print('计算各种归一化矩阵...')
TCM_row          = row_norm(TCM)
TCM_col          = col_norm(TCM)
TCM_bi, n_iter   = bi_norm(TCM)

print(f'Bi-norm 收敛于 {n_iter} 次迭代')
print(f'行和误差: {abs(TCM_bi.sum(1)-1).max():.1e}')
print(f'列和误差: {abs(TCM_bi.sum(0)-1).max():.1e}')

## 5. 四矩阵可视化

In [ ]:
short_names = [t.replace('_', ' ')[:20] for t in TECHNIQUES]

fig, axes = plt.subplots(2, 2, figsize=(22, 20))
fig.suptitle(f'Confusion Matrix Analysis — {LANG.upper()} / {METHOD}', fontsize=14, fontweight='bold')

def plot_cm(M, title, ax, vmax=None):
    im = ax.imshow(M, cmap='Blues', vmin=0, vmax=vmax or M.max(), aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    ax.set_xticks(range(C)); ax.set_xticklabels(short_names, rotation=90, fontsize=7)
    ax.set_yticks(range(C)); ax.set_yticklabels(short_names, fontsize=7)
    ax.set_xlabel('Predicted (ŷ)', fontsize=10)
    ax.set_ylabel('True Gold (y)', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')

plot_cm(TCM,     'Raw TCM',              axes[0, 0])
plot_cm(TCM_row, 'Row-norm (Recall)',    axes[0, 1], vmax=1)
plot_cm(TCM_col, 'Col-norm (Precision)', axes[1, 0], vmax=1)
plot_cm(TCM_bi,  'Bi-norm ★',           axes[1, 1], vmax=1)

plt.tight_layout()
save_path = BASE / f'results/TCM/cm_four_{LANG}_{METHOD}.png'
fig.savefig(save_path, dpi=110, bbox_inches='tight')
plt.show()
print(f'已保存: {save_path}')

## 6. Top 混淆技术对（Bi-norm非对角项）

In [ ]:
def get_top_confusions(Mbi, techniques, top_k=20):
    """
    从Bi-normalized TCM提取top-k对称混淆对
    按 max(A→B, B→A) 排序
    """
    C = len(techniques)
    seen, rows = set(), []
    # 按非对角项大小降序扫描
    for flat_idx in np.argsort(Mbi.ravel())[::-1]:
        i, j = divmod(flat_idx, C)
        if i == j: continue
        pair = tuple(sorted([techniques[i], techniques[j]]))
        if pair in seen: continue
        seen.add(pair)
        a, b   = pair
        ia, ib = techniques.index(a), techniques.index(b)
        a2b    = float(Mbi[ia, ib])
        b2a    = float(Mbi[ib, ia])
        rows.append({
            'Tech_A': a, 'Tech_B': b,
            'max_confusion': round(max(a2b, b2a), 5),
            'A→B':           round(a2b, 5),
            'B→A':           round(b2a, 5),
            'sum':           round(a2b + b2a, 5),
        })
        if len(rows) >= top_k: break
    return pd.DataFrame(rows)


top_conf = get_top_confusions(TCM_bi, TECHNIQUES, top_k=20)
print(f'=== Top 20 混淆技术对 — {LANG.upper()} / {METHOD} ===')
print(top_conf.to_string(index=False))

In [ ]:
# 条形图：分解A→B和B→A方向
t10 = top_conf.head(10).reset_index(drop=True)
x   = np.arange(len(t10)); w = 0.35
labels = [f"{r['Tech_A'].replace('_',' ')[:18]}\n↔\n{r['Tech_B'].replace('_',' ')[:18]}"
          for _, r in t10.iterrows()]

fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(x - w/2, t10['A→B'], w, label='A→B（A被误判为B）', color='steelblue', alpha=0.85)
ax.bar(x + w/2, t10['B→A'], w, label='B→A（B被误判为A）', color='tomato',    alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('Bi-normalized Confusion Score', fontsize=11)
ax.set_title(f'Top 10 Confusion Pairs — {LANG.upper()} / {METHOD}\n(higher score = harder to distinguish)', fontsize=12)
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(BASE / f'results/TCM/top_conf_{LANG}_{METHOD}.png', dpi=110, bbox_inches='tight')
plt.show()

## 7. 每类 P / R / F1

In [ ]:
def per_class_metrics(T, techniques):
    """从 Raw TCM 计算每类Precision/Recall/F1（与标准定义一致）"""
    rows = []
    for k, t in enumerate(techniques):
        tp  = T[k, k]
        row = T[k, :].sum()   # gold 总量（行和）
        col = T[:, k].sum()   # pred 总量（列和）
        R  = tp / row if row > 0 else 0.
        P  = tp / col if col > 0 else 0.
        F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.
        rows.append({'technique': t,
                     'Recall': round(R, 4), 'Precision': round(P, 4),
                     'F1': round(F1, 4),
                     'gold_count': round(row, 1), 'pred_count': round(col, 1)})
    return pd.DataFrame(rows).sort_values('F1')


metrics = per_class_metrics(TCM, TECHNIQUES)
print(f'TCM Macro F1 = {metrics["F1"].mean():.4f}')
print()
print(metrics.to_string(index=False))

# ── 在这里接着加 ──────────────────────────────────────
# 过滤掉 gold=0 的技术（避免伪混淆污染结果）
valid_techs = metrics[metrics['gold_count'] > 0]['technique'].tolist()
top_conf_filtered = top_conf[
    top_conf['Tech_A'].isin(valid_techs) &
    top_conf['Tech_B'].isin(valid_techs)
].reset_index(drop=True)

print(f'\n过滤后Top混淆对（排除gold=0技术，共{len(top_conf_filtered)}对）:')
print(top_conf_filtered.head(10).to_string(index=False))

## 8. Prompt 改进建议报告

In [ ]:
print('=' * 72)
print(f'Prompt-A 改进建议 — {LANG.upper()} / {METHOD}')
print(f'（基于 Bi-normalized TCM 的 Top 混淆对）')
print('=' * 72)

for rank, (_, row) in enumerate(top_conf_filtered.head(5).iterrows(), 1):
    a, b   = row['Tech_A'], row['Tech_B']
    a2b    = row['A→B']
    b2a    = row['B→A']
    score  = row['max_confusion']

    f1_a = metrics.loc[metrics['technique'] == a, 'F1'].values
    f1_b = metrics.loc[metrics['technique'] == b, 'F1'].values

    # 主要混淆方向（哪个更容易被误判成哪个）
    if a2b >= b2a:
        main_dir = f'{a}  →被误判为→  {b}'
    else:
        main_dir = f'{b}  →被误判为→  {a}'

    print(f'\n【第 {rank} 对】  max_confusion={score:.4f}  sum={row["sum"]:.4f}')
    print(f'  ● {a}  (TCM-F1={f1_a[0] if len(f1_a) else 0:.3f})')
    print(f'  ● {b}  (TCM-F1={f1_b[0] if len(f1_b) else 0:.3f})')
    print(f'  主要混淆方向: {main_dir}')
    print(f'  方向明细: {a}→{b} = {a2b:.4f}  |  {b}→{a} = {b2a:.4f}')
    print(f'  ─────────────────────────────────────────────────────────────')
    print(f'  ⚡ 建议在 Prompt-A 中添加（放在该技术的定义之后）:')
    print(f'''
  ## How to distinguish {a} from {b}:
  - {a}: [填写：核心定义特征，1-2句]
  - {b}: [填写：核心定义特征，1-2句]
  - KEY DIFFERENCE: [填写：最关键的区分点]
  - If the text shows X → label as {a}; if it shows Y → label as {b}
''')

print('=' * 72)

# Save results
out_conf    = BASE / f'results/TCM/confusion_top20_{LANG}_{METHOD}.csv'
out_metrics = BASE / f'results/TCM/per_class_{LANG}_{METHOD}.csv'
top_conf.to_csv(out_conf,    index=False)
metrics.to_csv(out_metrics,  index=False)
print(f'已保存: {out_conf.name},  {out_metrics.name}')

## 9. 跨语言批量分析

每种语言有独立文件夹：
```
results/TCM/
  en/  →  top15_*.csv, metrics_*.csv, robust_en.csv, figures/
  pl/  →  ...
  ru/  →  ...
  robust_all.csv  →  全局汇总
```

In [ ]:
# ── Cell 9.1：建目录 + 批量运行（9组实验）──────────────────────
from collections import Counter

# 在 Cell 9.1 的 from collections import Counter 之后加
short_names = [t.replace('_', ' ')[:18] for t in TECHNIQUES]
short_name  = lambda t: t.replace('_', ' ')[:22]

# 建各语言文件夹
for lang in ['en', 'pl', 'ru']:
    (BASE / f'results/TCM/{lang}/figures').mkdir(parents=True, exist_ok=True)
print('目录结构已创建：results/TCM/{en|pl|ru}/figures/')

all_tops    = {}   # (lang, method) → filtered top_conf DataFrame
all_metrics = {}   # (lang, method) → metrics DataFrame
all_tcm_bi  = {}   # (lang, method) → bi-normalized TCM matrix

for lang in ['en', 'pl', 'ru']:
    for method in ['sup_ft', 'prompt_a', 'iter_ens']:
        key = (lang, method)
        pf  = PRED_FILES.get(key)

        if pf is None or not pf.exists():
            print(f'[{lang}/{method}] 跳过（文件不存在）')
            continue

        print(f'处理 {lang}/{method}...')
        g = load_gold_with_zero(lang, BASE)
        p = load_spans_file(pf, TECH_SET)

        if len(set(g) & set(p)) == 0:
            print(f'  ⚠️ article_id 不对齐，跳过')
            continue

        yg, yp, _ = build_multihot(g, p, TECHNIQUES, TECH2IDX)
        T         = compute_tcm(yg, yp, 'one')
        Tb, _     = bi_norm(T)

        # 过滤 gold=0 技术
        _m     = per_class_metrics(T, TECHNIQUES)
        _valid = _m[_m['gold_count'] > 0]['technique'].tolist()
        _top   = get_top_confusions(Tb, TECHNIQUES, top_k=15)
        _top_f = _top[
            _top['Tech_A'].isin(_valid) & _top['Tech_B'].isin(_valid)
        ].reset_index(drop=True)

        all_tops[key]    = _top_f
        all_metrics[key] = _m
        all_tcm_bi[key]  = Tb

        # 保存到各语言文件夹
        _top_f.to_csv(BASE / f'results/TCM/{lang}/top15_{method}.csv',  index=False)
        _m.to_csv(    BASE / f'results/TCM/{lang}/metrics_{method}.csv', index=False)
        print(f'  ✅ {len(_top_f)} 有效混淆对 → results/TCM/{lang}/')

print(f'\n全部完成，共 {len(all_tops)} 组实验')

In [ ]:
# ── Cell 9.2：EN 可视化 ──────────────────────────────────────
lang = 'en'
lang_keys = [(lang, m) for m in ['sup_ft', 'prompt_a', 'iter_ens'] if (lang, m) in all_tops]
# short_names = [t.replace('_', ' ')[:18] for t in TECHNIQUES]
# short_name  = lambda t: t.replace('_', ' ')[:22]

# 9.2a：Top-5 条形图（3方法对比）
fig, axes = plt.subplots(1, len(lang_keys), figsize=(7*len(lang_keys), 5))
if len(lang_keys) == 1: axes = [axes]
fig.suptitle(f'Top Top Confusion Pairs — {lang.upper()}', fontsize=13, fontweight='bold')

for ax, key in zip(axes, lang_keys):
    df = all_tops[key].head(5).reset_index(drop=True)
    x  = np.arange(len(df)); w = 0.35
    ax.bar(x-w/2, df['A→B'], w, label='A→B', color='steelblue', alpha=0.85)
    ax.bar(x+w/2, df['B→A'], w, label='B→A', color='tomato',    alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{short_name(r['Tech_A'])}\n↔\n{short_name(r['Tech_B'])}"
                        for _, r in df.iterrows()], fontsize=7.5)
    ax.set_title(key[1], fontsize=11)
    ax.set_ylabel('Bi-norm Score')
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(BASE / f'results/TCM/{lang}/figures/top_conf_3methods.png', dpi=120, bbox_inches='tight')
plt.show()

# 9.2b：Bi-norm 热力图（3方法对比）
fig, axes = plt.subplots(1, len(lang_keys), figsize=(9*len(lang_keys), 8))
if len(lang_keys) == 1: axes = [axes]
fig.suptitle(f'Bi-normalized TCM — {lang.upper()}', fontsize=13, fontweight='bold')

for ax, key in zip(axes, lang_keys):
    Tb = all_tcm_bi[key]
    im = ax.imshow(Tb, cmap='Blues', vmin=0, vmax=0.5, aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.04)
    ax.set_xticks(range(C)); ax.set_xticklabels(short_names, rotation=90, fontsize=6)
    ax.set_yticks(range(C)); ax.set_yticklabels(short_names, fontsize=6)
    ax.set_title(key[1], fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.tight_layout()
fig.savefig(BASE / f'results/TCM/{lang}/figures/heatmap_3methods.png', dpi=110, bbox_inches='tight')
plt.show()

# 9.2c：EN 跨方法稳健混淆对
cnt_en = Counter()
for key in lang_keys:
    for _, r in all_tops[key].iterrows():
        cnt_en[(r['Tech_A'], r['Tech_B'])] += 1

print(f'── EN 跨方法稳健混淆对（{len(lang_keys)} 种方法）──')
for pair, c in cnt_en.most_common():
    mark = '★' if c == len(lang_keys) else (' ' if c == 1 else '△')
    print(f'  {mark} [{c}/{len(lang_keys)}]  {pair[0]}  <->  {pair[1]}')

pd.DataFrame([{'Tech_A': p[0], 'Tech_B': p[1], 'count': c, 'ratio': f'{c}/{len(lang_keys)}'}
              for p, c in cnt_en.most_common()
             ]).to_csv(BASE / f'results/TCM/{lang}/robust_{lang}.csv', index=False)
print(f'\n已保存: results/TCM/{lang}/robust_{lang}.csv')
print(f'已保存: results/TCM/{lang}/figures/top_conf_3methods.png')
print(f'已保存: results/TCM/{lang}/figures/heatmap_3methods.png')

In [ ]:
# ── Cell 9.3：PL 可视化 ──────────────────────────────────────
lang = 'pl'
lang_keys = [(lang, m) for m in ['sup_ft', 'prompt_a', 'iter_ens'] if (lang, m) in all_tops]

# Top-5 条形图
fig, axes = plt.subplots(1, len(lang_keys), figsize=(7*len(lang_keys), 5))
if len(lang_keys) == 1: axes = [axes]
fig.suptitle(f'Top Top Confusion Pairs — {lang.upper()}', fontsize=13, fontweight='bold')

for ax, key in zip(axes, lang_keys):
    df = all_tops[key].head(5).reset_index(drop=True)
    x  = np.arange(len(df)); w = 0.35
    ax.bar(x-w/2, df['A→B'], w, label='A→B', color='steelblue', alpha=0.85)
    ax.bar(x+w/2, df['B→A'], w, label='B→A', color='tomato',    alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{short_name(r['Tech_A'])}\n↔\n{short_name(r['Tech_B'])}"
                        for _, r in df.iterrows()], fontsize=7.5)
    ax.set_title(key[1], fontsize=11)
    ax.set_ylabel('Bi-norm Score')
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(BASE / f'results/TCM/{lang}/figures/top_conf_3methods.png', dpi=120, bbox_inches='tight')
plt.show()

# Bi-norm 热力图
fig, axes = plt.subplots(1, len(lang_keys), figsize=(9*len(lang_keys), 8))
if len(lang_keys) == 1: axes = [axes]
fig.suptitle(f'Bi-normalized TCM — {lang.upper()}', fontsize=13, fontweight='bold')

for ax, key in zip(axes, lang_keys):
    Tb = all_tcm_bi[key]
    im = ax.imshow(Tb, cmap='Blues', vmin=0, vmax=0.5, aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.04)
    ax.set_xticks(range(C)); ax.set_xticklabels(short_names, rotation=90, fontsize=6)
    ax.set_yticks(range(C)); ax.set_yticklabels(short_names, fontsize=6)
    ax.set_title(key[1], fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.tight_layout()
fig.savefig(BASE / f'results/TCM/{lang}/figures/heatmap_3methods.png', dpi=110, bbox_inches='tight')
plt.show()

# PL 跨方法稳健混淆对
cnt_pl = Counter()
for key in lang_keys:
    for _, r in all_tops[key].iterrows():
        cnt_pl[(r['Tech_A'], r['Tech_B'])] += 1

print(f'── PL 跨方法稳健混淆对（{len(lang_keys)} 种方法）──')
for pair, c in cnt_pl.most_common():
    mark = '★' if c == len(lang_keys) else (' ' if c == 1 else '△')
    print(f'  {mark} [{c}/{len(lang_keys)}]  {pair[0]}  <->  {pair[1]}')

pd.DataFrame([{'Tech_A': p[0], 'Tech_B': p[1], 'count': c, 'ratio': f'{c}/{len(lang_keys)}'}
              for p, c in cnt_pl.most_common()
             ]).to_csv(BASE / f'results/TCM/{lang}/robust_{lang}.csv', index=False)
print(f'\n已保存: results/TCM/{lang}/robust_{lang}.csv')
print(f'已保存: results/TCM/{lang}/figures/top_conf_3methods.png')
print(f'已保存: results/TCM/{lang}/figures/heatmap_3methods.png')

In [ ]:
# ── Cell 9.4：RU 可视化 ──────────────────────────────────────
lang = 'ru'
lang_keys = [(lang, m) for m in ['sup_ft', 'prompt_a', 'iter_ens'] if (lang, m) in all_tops]

# Top-5 条形图
fig, axes = plt.subplots(1, len(lang_keys), figsize=(7*len(lang_keys), 5))
if len(lang_keys) == 1: axes = [axes]
fig.suptitle(f'Top Top Confusion Pairs — {lang.upper()}', fontsize=13, fontweight='bold')

for ax, key in zip(axes, lang_keys):
    df = all_tops[key].head(5).reset_index(drop=True)
    x  = np.arange(len(df)); w = 0.35
    ax.bar(x-w/2, df['A→B'], w, label='A→B', color='steelblue', alpha=0.85)
    ax.bar(x+w/2, df['B→A'], w, label='B→A', color='tomato',    alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{short_name(r['Tech_A'])}\n↔\n{short_name(r['Tech_B'])}"
                        for _, r in df.iterrows()], fontsize=7.5)
    ax.set_title(key[1], fontsize=11)
    ax.set_ylabel('Bi-norm Score')
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(BASE / f'results/TCM/{lang}/figures/top_conf_3methods.png', dpi=120, bbox_inches='tight')
plt.show()

# Bi-norm 热力图
fig, axes = plt.subplots(1, len(lang_keys), figsize=(9*len(lang_keys), 8))
if len(lang_keys) == 1: axes = [axes]
fig.suptitle(f'Bi-normalized TCM — {lang.upper()}', fontsize=13, fontweight='bold')

for ax, key in zip(axes, lang_keys):
    Tb = all_tcm_bi[key]
    im = ax.imshow(Tb, cmap='Blues', vmin=0, vmax=0.5, aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.04)
    ax.set_xticks(range(C)); ax.set_xticklabels(short_names, rotation=90, fontsize=6)
    ax.set_yticks(range(C)); ax.set_yticklabels(short_names, fontsize=6)
    ax.set_title(key[1], fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.tight_layout()
fig.savefig(BASE / f'results/TCM/{lang}/figures/heatmap_3methods.png', dpi=110, bbox_inches='tight')
plt.show()

# RU 跨方法稳健混淆对
cnt_ru = Counter()
for key in lang_keys:
    for _, r in all_tops[key].iterrows():
        cnt_ru[(r['Tech_A'], r['Tech_B'])] += 1

print(f'── RU 跨方法稳健混淆对（{len(lang_keys)} 种方法）──')
for pair, c in cnt_ru.most_common():
    mark = '★' if c == len(lang_keys) else (' ' if c == 1 else '△')
    print(f'  {mark} [{c}/{len(lang_keys)}]  {pair[0]}  <->  {pair[1]}')

pd.DataFrame([{'Tech_A': p[0], 'Tech_B': p[1], 'count': c, 'ratio': f'{c}/{len(lang_keys)}'}
              for p, c in cnt_ru.most_common()
             ]).to_csv(BASE / f'results/TCM/{lang}/robust_{lang}.csv', index=False)
print(f'\n已保存: results/TCM/{lang}/robust_{lang}.csv')
print(f'已保存: results/TCM/{lang}/figures/top_conf_3methods.png')
print(f'已保存: results/TCM/{lang}/figures/heatmap_3methods.png')

In [ ]:
# ── Cell 9.5：全语言汇总 ─────────────────────────────────────
total    = len(all_tops)
cnt_all  = Counter()
for res in all_tops.values():
    for _, r in res.iterrows():
        cnt_all[(r['Tech_A'], r['Tech_B'])] += 1

print(f'=== 全语言×全方法稳健混淆对（共 {total} 组实验）===')
print(f'{"出现次数":>6}  {"技术对"}')
print('-' * 65)
for pair, c in cnt_all.most_common(20):
    mark = '★★' if c >= 5 else ('★ ' if c >= 3 else '  ')
    print(f'  {mark} [{c:2d}/{total}]  {pair[0]}  <->  {pair[1]}')

robust_all = [(p, c) for p, c in cnt_all.most_common() if c >= 3]
print(f'\n★ 高稳健混淆对（出现≥3次）: {len(robust_all)} 对')

pd.DataFrame([{'Tech_A': p[0], 'Tech_B': p[1], 'count': c, 'ratio': f'{c}/{total}'}
              for p, c in cnt_all.most_common()
             ]).to_csv(BASE / 'results/TCM/robust_all.csv', index=False)
print('已保存: results/TCM/robust_all.csv')

# 文件清单
print('\n── 所有输出文件 ──')
import os
for root, dirs, files in os.walk(BASE / 'results/TCM'):
    dirs.sort()
    for fname in sorted(files):
        fp = Path(root) / fname
        print(f'  {fp.relative_to(BASE)}')

In [ ]:
# ===== 公用：短码表 + 方法名 + 输出目录 =====
import matplotlib.pyplot as plt, numpy as np
from matplotlib import gridspec

SUPP = BASE / 'results/TCM/figures_supp'; SUPP.mkdir(parents=True, exist_ok=True)

ABBR = {
    'Appeal_to_Authority':'AtAu','Appeal_to_Popularity':'AtPo','Appeal_to_Values':'AtVa',
    'Appeal_to_Fear-Prejudice':'AtFe','Flag_Waving':'FW','Causal_Oversimplification':'CauO',
    'False_Dilemma-No_Choice':'FDil','Consequential_Oversimplification':'ConO','Straw_Man':'SM',
    'Red_Herring':'RH','Whataboutism':'Wh','Slogans':'Sl','Appeal_to_Time':'AtTi',
    'Conversation_Killer':'CK','Loaded_Language':'LL','Repetition':'Rep',
    'Exaggeration-Minimisation':'ExM','Obfuscation-Vagueness-Confusion':'OVC',
    'Name_Calling-Labeling':'NCL','Doubt':'Db','Guilt_by_Association':'GbA',
    'Appeal_to_Hypocrisy':'AtHy','Questioning_the_Reputation':'QtR',
}
def abbr(t):
    if t in ABBR: return ABBR[t]
    caps=''.join(ch for ch in t.replace('_',' ').title() if ch.isupper())
    return caps if len(caps)>=2 else t.replace('_',' ')[:4]

CODES = [abbr(t) for t in TECHNIQUES]
MNAME = {'sup_ft':'Sup-FT','prompt_a':'Prompt-A','iter_ens':'Iter-Ens'}
LNAME = {'en':'EN','pl':'PL','ru':'RU'}
LANGS, METHODS = ['en','pl','ru'], ['sup_ft','prompt_a','iter_ens']
VMAX = 0.5   # 与原图/论文一致；RU 峰值 0.983 会饱和为最深色（跨 panel 可比优先）
print('短码就绪，共', len(CODES), '个技术')

In [ ]:
# ===== 图 1：supplementary 9-panel 热力图（3语言 × 3方法）=====
rows = [l for l in LANGS if any((l,m) in all_tcm_bi for m in METHODS)]
cols = METHODS
fig = plt.figure(figsize=(13, 4.3*len(rows)))
gs  = gridspec.GridSpec(len(rows), len(cols)+1, width_ratios=[1]*len(cols)+[0.04], wspace=0.12, hspace=0.18)
im = None
for r,l in enumerate(rows):
    for c,m in enumerate(cols):
        ax = fig.add_subplot(gs[r,c])
        if (l,m) not in all_tcm_bi:
            ax.axis('off'); continue
        im = ax.imshow(all_tcm_bi[(l,m)], cmap='Blues', vmin=0, vmax=VMAX, aspect='equal')
        if r==0: ax.set_title(MNAME[m], fontsize=12, fontweight='bold')
        # 只在底行显 x 标签、最左列显 y 标签，减拥挤
        ax.set_xticks(range(C))
        ax.set_xticklabels(CODES if r==len(rows)-1 else ['']*C, rotation=90, fontsize=5)
        ax.set_yticks(range(C))
        ax.set_yticklabels(CODES if c==0 else ['']*C, fontsize=5)
        if c==0: ax.set_ylabel(f'{LNAME[l]}\nTrue', fontsize=11, fontweight='bold')
        if r==len(rows)-1: ax.set_xlabel('Predicted', fontsize=9)
cax = fig.add_subplot(gs[:,-1]); plt.colorbar(im, cax=cax, label='Bi-normalized weight')
fig.suptitle('Bi-normalized TCM across languages and methods', fontsize=14, fontweight='bold', y=0.995)
fig.savefig(SUPP/'supp_heatmap_9panel.pdf', bbox_inches='tight')
fig.savefig(SUPP/'supp_heatmap_9panel.png', dpi=300, bbox_inches='tight')
plt.show(); print('✅ supp_heatmap_9panel 已存')

In [ ]:
# ===== 图 2：正文备选 — 只 Sup-FT，横排 EN/PL/RU =====
fig = plt.figure(figsize=(12, 4.4))
gs  = gridspec.GridSpec(1, len(LANGS)+1, width_ratios=[1]*len(LANGS)+[0.04], wspace=0.10)
im = None
for c,l in enumerate(LANGS):
    ax = fig.add_subplot(gs[0,c])
    if (l,'sup_ft') not in all_tcm_bi: ax.axis('off'); continue
    im = ax.imshow(all_tcm_bi[(l,'sup_ft')], cmap='Blues', vmin=0, vmax=VMAX, aspect='equal')
    ax.set_title(LNAME[l], fontsize=12, fontweight='bold')
    ax.set_xticks(range(C)); ax.set_xticklabels(CODES, rotation=90, fontsize=5)
    ax.set_yticks(range(C)); ax.set_yticklabels(CODES if c==0 else ['']*C, fontsize=5)
    ax.set_xlabel('Predicted', fontsize=9)
    if c==0: ax.set_ylabel('True', fontsize=10)
cax = fig.add_subplot(gs[0,-1]); plt.colorbar(im, cax=cax, label='Bi-normalized weight')
fig.suptitle('Bi-normalized TCM (Sup-FT): PL cleanest, RU noisiest', fontsize=12.5, fontweight='bold', y=1.02)
fig.savefig(SUPP/'fig_tcm_supft_row.pdf', bbox_inches='tight')
fig.savefig(SUPP/'fig_tcm_supft_row.png', dpi=300, bbox_inches='tight')
plt.show(); print('✅ fig_tcm_supft_row 已存')

In [ ]:
# ===== 图 3：清理版 Top-5 混淆对 bar（逐语言）=====
for l in LANGS:
    keys = [(l,m) for m in METHODS if (l,m) in all_tops]
    if not keys: continue
    fig, axes = plt.subplots(1, len(keys), figsize=(6.2*len(keys), 4.6))
    if len(keys)==1: axes=[axes]
    fig.suptitle(f'Top-5 Confusion Pairs — {LNAME[l]}', fontsize=13, fontweight='bold')  # 修 Top Top
    for ax,key in zip(axes,keys):
        df = all_tops[key].head(5).reset_index(drop=True)
        x=np.arange(len(df)); w=0.38
        ax.bar(x-w/2, df['A\u2192B'], w, label='A\u2192B', color='#4c72b0', alpha=0.88, edgecolor='white')
        ax.bar(x+w/2, df['B\u2192A'], w, label='B\u2192A', color='#dd8452', alpha=0.88, edgecolor='white')
        ax.set_xticks(x)
        ax.set_xticklabels([f"{abbr(r['Tech_A'])} \u2194 {abbr(r['Tech_B'])}" for _,r in df.iterrows()],
                           rotation=30, ha='right', fontsize=9)
        ax.set_title(MNAME[key[1]], fontsize=11)
        ax.set_ylabel('Bi-norm score'); ax.legend(fontsize=8, frameon=False); ax.grid(axis='y', alpha=0.3)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.tight_layout()
    fig.savefig(SUPP/f'supp_confpairs_{l}.pdf', bbox_inches='tight')
    fig.savefig(SUPP/f'supp_confpairs_{l}.png', dpi=300, bbox_inches='tight')
    plt.show()
print('✅ supp_confpairs_{en,pl,ru} 已存')

In [ ]:
# ===== 生成图注用的短码对照表（粘进 supplementary caption）=====
pairs = [f'{abbr(t)}={t.replace("_"," ")}' for t in TECHNIQUES]
print('Technique codes: ' + '; '.join(pairs) + '.')